# Generating bases of Slater determinats and Configuration State Functions (CSFs) as well as corresponding transformation matrices

In this super tutorial, we will learn how to use the Libra's built-in functions to compute the overlap integrals using different methods.

## Table of Content: <a name="TOC"></a>

1. [Generating Slater Determinants with Permutation Parity - `slatdet` module](#1)

   1.1. [Active space and spin-orbitals](#1.1)
   
   1.2. [Slater determinants as ordered spin-orbital tuples](#1.2)
   
   1.3. [Canonical ordering of spin-orbitals](#1.3)
   
   1.4. [Fermionic antisymmetry and permutation parity](#1.4)
   
   1.5. [Enumeration strategy](#1.5)
   
   1.6. [Example](#1.6)
   
2. [Building a Minimal CSF Basis from Raw Configurations - `interfaces` module](#2)

   2.1. [Physical motivation](#2.1)
   
   2.2. [Input configurations (raw form)](#2.2)
   
   2.3. [Active space restriction](#2.3)
   
   2.4. [Electron number constraint](#2.4)
   
   2.5. [Spin constraint via number of unpaired electrons](#2.5)
   
   2.6. [Expansion into matching determinants](#2.6)
   
   2.7. [Canonical ordering and fermionic phase](#2.7)
   
   2.8. [Minimal CSF basis](#2.8)   

   2.9. [Transposed basis representation](#2.9)
   
   2.10. [Worked example](#2.10)
   
3. [Mapping Configurations and Constructing the CSF Transformation Matrix](#3)

   3.1. [Purpose of the function](#3.1)
   
   3.2. [Raw input configurations](#3.2)
   
   3.3. [Minimal determinant basis in the active space](#3.3)
   
   3.4. [Mapping to the target orbital space](#3.4)
   
   3.5. [Spin quantum numbers and CSFs](#3.5)
   
   3.6. [Construction of the transformation matrix](#3.6)
   
   3.7. [Interpretation of the output](#3.7)
   
   3.8. [Example interpretation](#3.8)
   
4. [Mapping Configurations and Constructing the CSF Transformation Matrix](#4)

   4.1. [System definition (toy model)](#4.1)
   
   4.2. [Determinant basis (fixed 𝑀𝑠=0)](#4.2)
   
   4.3. [Spin-adapted CSFs](#4.3)
   
   4.4. [Summary of bases](#4.4)
   
   4.5. [Configuration-to-CSF transformation matrix 𝑇](#4.5)
   
   4.6. [Connection to `configs_and_T_matrix`](#4.6)

## A. Learning objectives

- To understand the underlying algorithm in step 2 and 3 calculations
- To be able to use the functions used in step 2 and 3 modules
- To be able to debug the Libra code for overlap calculations using the methods presented in this tutorial
- To be able to implement new algorithms and modifying the underlying code related to step 2 and 3


## B. Use cases

- Manually construct a Slater Determinant basis
- Auto-generate a Slater Determinant basis
- Constructing configuration spin functions
- Spin-adaptation of Slater determinants

## C. Functions

- `libra_py`
  - `citools`
    - `interfaces`
      - [`configs_and_T_matrix`](#configs_and_T_matrix-1)
      - [`configs_and_T_matrix_singlet`](#configs_and_T_matrix_singlet-1)
    - `slatdet`
      - [`build_minimal_csf_basis`](#build_minimal_csf_basis-1)
      - [`build_minimal_csf_basis_singlet`](#build_minimal_csf_basis_singlet-1)
      - [`generate_single_excitations`](#generate_single_excitations-1)
      - [`generate_determinants_with_parity`](#generate_determinants_with_parity-1)
      

In [1]:
import numpy as np
import libra_py
import libra_py.citools.slatdet as sd
import libra_py.citools.interfaces as interfaces

<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for boost::python::detail::container_element<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, unsigned long, boost::python::detail::final_vector_derived_policies<std::vector<std::vector<int, std::allocator<int> >, std::allocator<std::vector<int, std::allocator<int> > > >, false> > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWarning: to-Python converter for std::vector<std::vector<float, std::allocator<float> >, std::allocator<std::vector<float, std::allocator<float> > > > already registered; second conversion method ignored.
<frozen importlib._bootstrap>:241: RuntimeWar

## 1. Generating Slater Determinants with Permutation Parity - `slatdet` module
[Back to TOC](#TOC)
<a name="1"></a>

It serves the purpose of generating Slater determinants (a list-of-integers representation) in a consistent way and was already overviewed in the previous tutorial 

The content of this module is as follows:

In [2]:
dir(sd)

['Any',
 'Generator',
 'List',
 'Sequence',
 'Tuple',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'annotations',
 'canonical_sort_key',
 'generate_determinants_with_parity',
 'generate_single_excitations',
 'itertools',
 'math',
 'np',
 'permutation_parity',
 'slater_overlap_matrix']

### 1.1. Active space and spin-orbitals
[Back to TOC](#TOC)
<a name="1.1"></a>

Let the **active space** be defined by a set of spatial orbitals

$$
\mathcal{A} = \{ i_1, i_2, \dots, i_M \}.
$$

Each spatial orbital $i \in \mathcal{A}$ generates **two spin-orbitals**:

$$
\chi_{i\alpha} \equiv +i,
\qquad
\chi_{i\beta} \equiv -i.
$$

The full spin-orbital set is therefore

$$
\mathcal{S} = \{ +i_1, -i_1, +i_2, -i_2, \dots, +i_M, -i_M \}.
$$


### 1.2. Slater determinants as ordered spin-orbital tuples
[Back to TOC](#TOC)
<a name="1.2"></a>

An $N$-electron Slater determinant is defined by selecting an **ordered tuple**
of $N$ distinct spin-orbitals

$$
\mathbf{d} = (p_1, p_2, \dots, p_N),
\qquad
p_k \in \mathcal{S},
$$

subject to the **Pauli principle**

$$
p_i \neq p_j \quad \text{for} \quad i \neq j.
$$

If `allow_double_occupancy = False`, an additional constraint is enforced:

$$
\{+i, -i\} \nsubseteq \mathbf{d}
\quad \forall i \in \mathcal{A},
$$

i.e., at most one spin-orbital per spatial orbital may be occupied.


### 1.3. Canonical ordering of spin-orbitals
[Back to TOC](#TOC)
<a name="1.3"></a>

To define Slater determinants uniquely, each determinant is transformed into a
**canonical order**. A typical choice is

$$
(-i) < (+i) < (-j) < (+j) \quad \text{for} \quad i < j,
$$

which is implemented via the helper function

`canonical_sort_key(spin_orbital)`.

The canonically ordered determinant is

$$
\mathbf{d}^{\text{can}} = \text{sort}(\mathbf{d}).
$$


### 1.4. Fermionic antisymmetry and permutation parity
[Back to TOC](#TOC)
<a name="1.4"></a>

A Slater determinant corresponds to the antisymmetrized product

$$
\lvert \mathbf{d} \rangle
=
\frac{1}{\sqrt{N!}}
\begin{vmatrix}
\chi_{p_1}(1) & \chi_{p_2}(1) & \cdots & \chi_{p_N}(1) \\
\chi_{p_1}(2) & \chi_{p_2}(2) & \cdots & \chi_{p_N}(2) \\
\vdots        & \vdots        & \ddots & \vdots        \\
\chi_{p_1}(N) & \chi_{p_2}(N) & \cdots & \chi_{p_N}(N)
\end{vmatrix}.
$$

Reordering the spin-orbitals introduces a fermionic sign:

$$
\lvert \mathbf{d} \rangle
=
\text{sgn}(\pi)\;
\lvert \mathbf{d}^{\text{can}} \rangle,
$$

where $\pi$ is the permutation mapping $\mathbf{d} \to \mathbf{d}^{\text{can}}$.

The **permutation parity**

$$
\text{sgn}(\pi) =
\begin{cases}
+1 & \text{even permutation}, \\
-1 & \text{odd permutation},
\end{cases}
$$

is computed by

`permutation_parity(original, sorted_tuple)`.


### 1.5. Enumeration strategy
[Back to TOC](#TOC)
<a name="1.5"></a>

The function proceeds as follows:

1. Construct the full spin-orbital list $\mathcal{S}$.
2. Enumerate all combinations of $N$ spin-orbitals from $\mathcal{S}$.
3. Discard combinations violating the Pauli principle.
4. Optionally discard determinants with double occupancy.
5. Sort each determinant into canonical order.
6. Compute the permutation parity.
7. Yield `(determinant, parity)`.

Mathematically, the output is the set

$$
\left\{
\left(
\mathbf{d}^{\text{can}},
\text{sgn}(\pi)
\right)
\;\middle|\;
\mathbf{d} \subset \mathcal{S},\;
|\mathbf{d}| = N
\right\}.
$$


### 1.6. Example
[Back to TOC](#TOC)
<a name="1.6"></a>

For

$$
\mathcal{A} = \{1, 2\}, \quad N = 2,
$$

the allowed spin-orbitals are

$$
\mathcal{S} = \{-2, -1, +1, +2\}.
$$

The function generates determinants such as

$$
(-2, +1), \quad \text{parity} = -1,
$$

indicating that the original ordering differs by an odd permutation from
canonical order.


Let's just do a brief reminder on the `generate_determinants_with_parity` function
<a name="generate_determinants_with_parity-1"></a>

In [3]:
help(sd.generate_determinants_with_parity)

Help on function generate_determinants_with_parity in module libra_py.citools.slatdet:

generate_determinants_with_parity(active_orbitals: 'List[int]', N: 'int', allow_double_occupancy: 'bool' = True) -> 'Generator[Tuple[Tuple[int, ...], int], None, None]'
    Generate all possible Slater determinants (configurations) from a given set of 
    active orbitals, including their permutation parities.
    
    Each spatial orbital in `active_orbitals` contributes two spin-orbitals: 
    α (represented by +i) and β (represented by -i). The function yields all 
    unique combinations of `N` spin-orbitals consistent with the Pauli principle 
    and, optionally, with or without double occupancy of the same spatial orbital.
    
    The permutation parity is computed relative to the canonically sorted order 
    of spin-orbitals, allowing subsequent antisymmetrization when constructing 
    configuration state functions (CSFs).
    
    Parameters
    ----------
    active_orbitals : list of i

In [4]:
list(sd.generate_determinants_with_parity([1, 2], 2))

[((1, -1), 1),
 ((1, 2), 1),
 ((1, -2), 1),
 ((-1, 2), 1),
 ((-1, -2), 1),
 ((2, -2), 1)]

In [5]:
list(sd.generate_determinants_with_parity([1, 2], 2, allow_double_occupancy=False))

[((1, 2), 1), ((1, -2), 1), ((-1, 2), 1), ((-1, -2), 1)]

In [6]:
list(sd.generate_determinants_with_parity([6, 7, 8, 9, 10, 11], 6, allow_double_occupancy=False))

[((6, 7, 8, 9, 10, 11), 1),
 ((6, 7, 8, 9, 10, -11), 1),
 ((6, 7, 8, 9, -10, 11), 1),
 ((6, 7, 8, 9, -10, -11), 1),
 ((6, 7, 8, -9, 10, 11), 1),
 ((6, 7, 8, -9, 10, -11), 1),
 ((6, 7, 8, -9, -10, 11), 1),
 ((6, 7, 8, -9, -10, -11), 1),
 ((6, 7, -8, 9, 10, 11), 1),
 ((6, 7, -8, 9, 10, -11), 1),
 ((6, 7, -8, 9, -10, 11), 1),
 ((6, 7, -8, 9, -10, -11), 1),
 ((6, 7, -8, -9, 10, 11), 1),
 ((6, 7, -8, -9, 10, -11), 1),
 ((6, 7, -8, -9, -10, 11), 1),
 ((6, 7, -8, -9, -10, -11), 1),
 ((6, -7, 8, 9, 10, 11), 1),
 ((6, -7, 8, 9, 10, -11), 1),
 ((6, -7, 8, 9, -10, 11), 1),
 ((6, -7, 8, 9, -10, -11), 1),
 ((6, -7, 8, -9, 10, 11), 1),
 ((6, -7, 8, -9, 10, -11), 1),
 ((6, -7, 8, -9, -10, 11), 1),
 ((6, -7, 8, -9, -10, -11), 1),
 ((6, -7, -8, 9, 10, 11), 1),
 ((6, -7, -8, 9, 10, -11), 1),
 ((6, -7, -8, 9, -10, 11), 1),
 ((6, -7, -8, 9, -10, -11), 1),
 ((6, -7, -8, -9, 10, 11), 1),
 ((6, -7, -8, -9, 10, -11), 1),
 ((6, -7, -8, -9, -10, 11), 1),
 ((6, -7, -8, -9, -10, -11), 1),
 ((-6, 7, 8, 9, 10, 11),

<a name="generate_single_excitations-1"></a>

In [7]:
help(sd.generate_single_excitations)

Help on function generate_single_excitations in module libra_py.citools.slatdet:

generate_single_excitations(active_orbitals: 'List[int]', nelec: 'int')
    Generate the ground-state determinant first, followed by all single-excitation
    determinants (sorted + parity), for a closed-shell reference inside an active space.
    
    Parameters
    ----------
    active_orbitals : list[int]
        Ordered list of spatial orbital numbers in the active space.
        Example: [1,2,3,4,5,6,7,8,9,10]
    
    nelec : int
        Number of electrons in the active space. Must be even for closed-shell.
        Example: 10 -> 5 occupied spatial orbitals (paired).
    
    Yields
    ------
    (det_sorted, parity)
        det_sorted: tuple of signed spin-orbitals (canonical order)
        parity: +1 or -1



In [8]:
for x,v in sd.generate_single_excitations([1,2,3], 2): print(x,v)

(1, -1) 1
(-1, 2) 1
(1, -2) 1
(-1, 3) 1
(1, -3) 1


In [9]:
for x,v in sd.generate_determinants_with_parity([1,2,3], 2): print(x,v)

(1, -1) 1
(1, 2) 1
(1, -2) 1
(1, 3) 1
(1, -3) 1
(-1, 2) 1
(-1, -2) 1
(-1, 3) 1
(-1, -3) 1
(2, -2) 1
(2, 3) 1
(2, -3) 1
(-2, 3) 1
(-2, -3) 1
(3, -3) 1


## 2. Building a Minimal CSF Basis from Raw Configurations - `interfaces` module
[Back to TOC](#TOC)
<a name="2"></a>

This module provides higher-level functions for generating bases of SDs and CSFs for excitations given by the user and generates the corresponding transformation matrices.

This is what it contains:

In [10]:
dir(interfaces)

['Any',
 'CMATRIX',
 'Counter',
 'Dict',
 'Iterable',
 'List',
 'MATRIX',
 'Tuple',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'build_minimal_csf_basis',
 'build_minimal_csf_basis_singlet',
 'conf2csf_matrix',
 'configs_and_T_matrix',
 'configs_and_T_matrix_singlet',
 'csf',
 'find_matches',
 'map_to_active_indices',
 'np',
 'sd']

Let's discuss the function

`build_minimal_csf_basis(configs, active_space, nelec, max_unpaired)`

which maps *raw configurations* (as produced by MOPAC / Libra) onto a **minimal,
spin-consistent determinant / CSF basis** inside a chosen active space.


### 2.1. Physical motivation
[Back to TOC](#TOC)
<a name="2.1"></a>

Electronic structure codes often output configurations in a **compressed or
non-unique form**, e.g.:

- unordered spin-orbitals
- incomplete spin resolution
- implicit phase conventions

However, **quantum dynamics, CI, and CSF constructions** require:

- explicitly antisymmetrized Slater determinants
- consistent spin constraints
- explicit fermionic phase tracking

This function performs the **projection**

$$
\text{raw configuration}
\;\longrightarrow\;
\text{minimal set of determinants consistent with spin and active space}
$$


### 2.2. Input configurations (raw form)
[Back to TOC](#TOC)
<a name="2.2"></a>

Each input configuration is a tuple of signed integers:

$$
\mathbf{c}_{\text{raw}} = (p_1, p_2, \dots, p_{N}),
$$

where:

- $|p_i|$ is a **spatial orbital index**
- $\operatorname{sign}(p_i)$ encodes spin:
  - $+i$ → $\alpha$
  - $-i$ → $\beta$

These configurations may:
- not be canonically ordered
- implicitly represent **multiple spin couplings**
- lack explicit antisymmetry phases


### 2.3. Active space restriction
[Back to TOC](#TOC)
<a name="2.3"></a>

The **active space** is defined as

$$
\mathcal{A} = \{ i_1, i_2, \dots, i_M \}.
$$

Only determinants satisfying

$$
|p_k| \in \mathcal{A}
\quad \forall k
$$

are considered.

This ensures that all resulting determinants live entirely within the chosen
active space.


### 2.4. Electron number constraint
[Back to TOC](#TOC)
<a name="2.4"></a>

The total number of electrons is fixed:

$$
N = \text{nelec}.
$$

All generated determinants must contain **exactly $N$ spin-orbitals**.


### 2.5. Spin constraint via number of unpaired electrons
[Back to TOC](#TOC)
<a name="2.5"></a>

The parameter

$$
\text{max_unpaired} = 2 M_s
$$

imposes a **spin projection constraint**.

For a determinant with:
- $N_\alpha$ alpha electrons
- $N_\beta$ beta electrons

the spin projection is

$$
M_s = \frac{1}{2}(N_\alpha - N_\beta).
$$

Thus, the allowed determinants must satisfy

$$
|N_\alpha - N_\beta| = \text{max_unpaired}.
$$

Example:
- `max_unpaired = 0` → singlets only
- `max_unpaired = 2` → doublets
- `max_unpaired = 4` → triplets

This filters out determinants with incompatible spin projections.


### 2.6. Expansion into matching determinants
[Back to TOC](#TOC)
<a name="2.6"></a>

A **single raw configuration** may correspond to **multiple valid determinants**
once spin consistency and antisymmetry are enforced.

Conceptually, the function performs:

$$
\mathbf{c}_{\text{raw}}
\;\longrightarrow\;
\left\{
(\mathbf{d}_1, \phi_1),
(\mathbf{d}_2, \phi_2),
\dots
\right\}
$$

where:
- $\mathbf{d}_k$ are canonically ordered determinants
- $\phi_k = \pm 1$ are fermionic phases

This accounts for:
- reordering of spin-orbitals
- equivalent spin couplings
- phase changes due to permutations


### 2.7. Canonical ordering and fermionic phase
[Back to TOC](#TOC)
<a name="2.7"></a>

Each determinant is sorted into a **canonical order**:

$$
\mathbf{d}^{\text{can}} = \text{sort}(\mathbf{d}),
$$

using a fixed convention (e.g. $\beta$ before $\alpha$, lower orbitals first).

Reordering introduces a sign:

$$
\lvert \mathbf{d} \rangle
=
\text{sgn}(\pi)\;
\lvert \mathbf{d}^{\text{can}} \rangle,
$$

where $\pi$ is the permutation mapping the raw ordering to the canonical one.

The returned **phase** is exactly

$$
\phi = \text{sgn}(\pi) \in \{+1, -1\}.
$$


### 2.8. Minimal CSF basis
[Back to TOC](#TOC)
<a name="2.8"></a>

The output `min_basis` is a list of pairs:

$$
\text{min_basis}
=
\left[
(\mathbf{d}^{\text{can}}_1, \phi_1),
(\mathbf{d}^{\text{can}}_2, \phi_2),
\dots
\right].
$$

Key properties:

- Only determinants **consistent with the raw configurations** are included
- Spin-incompatible determinants are excluded
- The basis is **minimal**: no redundant determinants
- Phases ensure correct antisymmetry

This basis can be interpreted as a **determinant-level representation of CSFs**.

### 2.9. Transposed basis representation
[Back to TOC](#TOC)
<a name="2.9"></a>

The second return value is:

$$
\text{transposed_basis} = \text{zip}(*\text{min_basis}),
$$

which yields:

- a tuple of all determinants
- a tuple of all phases

This is convenient for:
- vectorized overlap computations
- CI coefficient assembly
- determinant-to-CSF transformations


### 2.10. Worked example
[Back to TOC](#TOC)
<a name="2.10"></a>

Given:

$$
\mathbf{c}_{\text{raw}} = (6, -6, 7, -7, 9, -8),
$$

with:

- $\mathcal{A} = \{6,7,8,9,10,11\}$
- $N = 6$
- $M_s = 0$

the function identifies two valid singlet determinants:

$$
(6, -6, 7, -7, 8, -9),
\quad
(6, -6, 7, -7, -8, 9),
$$

both with phase $+1$.

These correspond to the two spin couplings consistent with the raw configuration
inside the active space.

<a name="build_minimal_csf_basis-1"></a>

In [11]:
help(interfaces.build_minimal_csf_basis)

Help on function build_minimal_csf_basis in module libra_py.citools.interfaces:

build_minimal_csf_basis(configs: List[Tuple[int, ...]], active_space: List[int], nelec: int, max_unpaired: int) -> Tuple[List[Tuple[Any, ...]], List[Tuple[Any, ...]]]
    Construct a minimal CSF (Configuration State Function) basis
    from raw MOPAC/Libra configurations.
    
    Parameters
    ----------
    configs : list[tuple[int]]
        Configurations extracted from Libra or MOPAC output (raw orbital numbers).
    active_space : list[int]
        List of active orbital numbers defining the active space.
    nelec : int
        Total number of electrons.
    max_unpaired : int
        Twice the target spin projection (2*Ms).
        For example, 0 corresponds to singlet-only configurations.
    
    Returns
    -------
    min_basis : list[tuple]
        List of matching determinants (CSFs) as (configuration, phase) tuples.
    transposed_basis : list[tuple]
        Transposed representation, i.e. `

In [12]:
# This means we only consider excitations within the given active space of 
# spatial orbitals 6,7,8,9,10, and 11
# in it we have 6 electrons, which means the orbitals 6, 7, and 8 are doubly-occupied
# in the ground states
# so the excitation could be from 8 to 9 (alpha-electron)

configs0_raw = [(6, -6, 7, -7, 9, -8)]
active_space = [6, 7, 8, 9, 10, 11]
nelec = 6
max_unpaired = 0 # singlets only

min_basis, (all_confs, all_phases) = interfaces.build_minimal_csf_basis(
    configs0_raw, active_space, nelec, max_unpaired)


print(min_basis)

print(all_confs)

print(all_phases)

[((6, -6, 7, -7, 8, -9), 1), ((6, -6, 7, -7, -8, 9), 1)]
((6, -6, 7, -7, 8, -9), (6, -6, 7, -7, -8, 9))
(1, 1)


In [13]:
# Or we can extend the active space, to also include spatial orbital 5, 
# but now we have to also increase the number of electrons to place on these
# orbitals to create an analog of the former excitations 
# so it means the orbitals 5, 6, 7, and 8 are doubly-occupied
# and orbitals 9, 10, and 11 are vacant in the ground states
# so the excitation could be from 8 to 9 (alpha-electron)

configs0_raw = [(5, -5, 6, -6, 7, -7, 9, -8)]
active_space = [5, 6, 7, 8, 9, 10, 11]
nelec = 8
max_unpaired = 0 # singlets only

min_basis, (all_confs, all_phases) = interfaces.build_minimal_csf_basis(
    configs0_raw, active_space, nelec, max_unpaired)


print(min_basis)

print(all_confs)

print(all_phases)

[((5, -5, 6, -6, 7, -7, 8, -9), 1), ((5, -5, 6, -6, 7, -7, -8, 9), 1)]
((5, -5, 6, -6, 7, -7, 8, -9), (5, -5, 6, -6, 7, -7, -8, 9))
(1, 1)


While this is a general function, under the hood it conducts a generation of an excessive list of excited Slater determinants and may be extremely expensive for determinants with many orbitals and electrons. 

<a name="build_minimal_csf_basis_singlet-1"></a>

In [14]:
help(interfaces.build_minimal_csf_basis_singlet)

Help on function build_minimal_csf_basis_singlet in module libra_py.citools.interfaces:

build_minimal_csf_basis_singlet(configs: List[Tuple[int, ...]], active_space: List[int], nelec: int, max_unpaired: int) -> Tuple[List[Tuple[Any, ...]], List[Tuple[Any, ...]]]
    Construct a minimal CSF (Configuration State Function) basis
    from raw MOPAC/Libra configurations.
    
    Parameters
    ----------
    configs : list[tuple[int]]
        Configurations extracted from Libra or MOPAC output (raw orbital numbers).
    active_space : list[int]
        List of active orbital numbers defining the active space.
    nelec : int
        Total number of electrons.
    max_unpaired : int
        Twice the target spin projection (2*Ms).
        For example, 0 corresponds to singlet-only configurations.
    
    Returns
    -------
    min_basis : list[tuple]
        List of matching determinants (CSFs) as (configuration, phase) tuples.
    transposed_basis : list[tuple]
        Transposed repres

In [15]:
configs0_raw = [(6, -6, 7, -7, 9, -8)]
active_space = [6, 7, 8, 9, 10, 11]
nelec = 6
max_unpaired = 0 # singlets only

min_basis, (all_confs, all_phases) = interfaces.build_minimal_csf_basis_singlet(
    configs0_raw, active_space, nelec, max_unpaired)


print(min_basis)

print(all_confs)

print(all_phases)

[((6, -6, 7, -7, -8, 9), 1), ((6, -6, 7, -7, 8, -9), 1)]
((6, -6, 7, -7, -8, 9), (6, -6, 7, -7, 8, -9))
(1, 1)


In [16]:
configs0_raw = [(5, -5, 6, -6, 7, -7, 9, -8)]
active_space = [5, 6, 7, 8, 9, 10, 11]
nelec = 8
max_unpaired = 0 # singlets only

min_basis, (all_confs, all_phases) = interfaces.build_minimal_csf_basis_singlet(
    configs0_raw, active_space, nelec, max_unpaired)


print(min_basis)

print(all_confs)

print(all_phases)

[((5, -5, 6, -6, 7, -7, -8, 9), 1), ((5, -5, 6, -6, 7, -7, 8, -9), 1)]
((5, -5, 6, -6, 7, -7, -8, 9), (5, -5, 6, -6, 7, -7, 8, -9))
(1, 1)


## 3. Mapping Configurations and Constructing the CSF Transformation Matrix
[Back to TOC](#TOC)
<a name="3"></a>

Let's discuss the function

`configs_and_T_matrix(configs0_raw, active_space, orbital_space, nelec, S, Ms)`

which constructs a **spin-adapted CAS basis** and the corresponding
**configuration-to-CSF transformation matrix**.


### 3.1. Purpose of the function
[Back to TOC](#TOC)
<a name="3.1"></a>

Electronic-structure and quantum-dynamics workflows often require two
simultaneous representations:

1. **Determinant (configuration) basis**
   — explicit spin-orbital occupations
2. **Spin-adapted CSF basis**
   — eigenfunctions of $\hat{S}^2$ and $\hat{S}_z$

This function builds the mapping

$$
\lvert \text{CSF}_I^{(S, M_s)} \rangle
=
\sum_{j}
T_{jI}
\lvert D_j \rangle,
$$

where:

- $\lvert D_j \rangle$ are **Slater determinants**
- $\lvert \text{CSF}_I \rangle$ are **spin-adapted configuration state functions**
- $T$ is the **configuration-to-CSF transformation matrix**


### 3.2. Raw input configurations
[Back to TOC](#TOC)
<a name="3.2"></a>

The input `configs0_raw` is a list of raw signed orbital configurations:

$$
\mathbf{c}_{\text{raw}} = (p_1, p_2, \dots, p_{N}),
$$

with:

- $|p_k|$ = spatial orbital index
- $\operatorname{sign}(p_k)$ = spin
  - $+i \rightarrow \alpha$
  - $-i \rightarrow \beta$

These configurations:
- may not be ordered
- may not be spin-adapted
- may represent compact CI outputs


### 3.3. Minimal determinant basis in the active space
[Back to TOC](#TOC)
<a name="3.3"></a>

Using the **active space**

$$
\mathcal{A} = \{ a_1, a_2, \dots, a_M \},
$$

the function first constructs the **minimal determinant basis**

$$
\{ \lvert D_j^{\text{act}} \rangle \},
$$

such that:

- all determinants contain exactly `nelec` electrons
- all spin-orbitals belong to $\mathcal{A}$
- determinants satisfy the spin projection constraint

$$
M_s = \frac{1}{2}(N_\alpha - N_\beta)
$$

This step removes redundant or spin-forbidden configurations.


### 3.4. Mapping to the target orbital space
[Back to TOC](#TOC)
<a name="3.4"></a>

The determinant basis is then **reindexed** to the requested orbital space

$$
\mathcal{O} = \{ o_1, o_2, \dots, o_K \}.
$$

Each spin-orbital is mapped via:

$$
\pm a_i \;\longrightarrow\; \pm k,
\quad
k = \text{index}(a_i \in \mathcal{O}).
$$

The resulting mapped determinants are

$$
\mathbf{d}_j^{\text{map}}
=
(\pm k_1, \pm k_2, \dots, \pm k_{N}),
$$

which allows consistent indexing when computing overlaps,
Hamiltonians, or time propagation in a **larger orbital basis**.


### 3.5. Spin quantum numbers and CSFs
[Back to TOC](#TOC)
<a name="3.5"></a>

The function enforces **total spin** constraints:

- Total spin quantum number:

$$
\hat{S}^2 \lvert \text{CSF}_I \rangle
=
S(S+1)\lvert \text{CSF}_I \rangle
$$

- Spin projection:

$$
\hat{S}_z \lvert \text{CSF}_I \rangle
=
M_s \lvert \text{CSF}_I \rangle
$$

Only determinants compatible with $M_s$ are included, and only linear
combinations yielding total spin $S$ are retained.


### 3.6. Construction of the transformation matrix
[Back to TOC](#TOC)
<a name="3.6"></a>

Let:

- $\{ \lvert D_j \rangle \}$ be the determinant basis
- $\{ \lvert \text{CSF}_I \rangle \}$ be the spin-adapted basis

The transformation matrix $T$ is defined by

$$
T_{jI}
=
\langle D_j \mid \text{CSF}_I \rangle.
$$

Thus:

- **Columns** of $T$ represent CSFs
- **Rows** of $T$ represent determinants
- $T$ is generally **complex-valued**

The CSFs are normalized:

$$
\sum_j |T_{jI}|^2 = 1.
$$


### 3.7. Interpretation of the output
[Back to TOC](#TOC)
<a name="3.7"></a>

#### 3.7.1 `mapped_basis`

A list of determinants:

$$
\text{mapped_basis}
=
\left\{
\mathbf{d}_1^{\text{map}},
\mathbf{d}_2^{\text{map}},
\dots
\right\},
$$

each expressed in the indexing of `orbital_space`.

These are suitable for:
- determinant overlap matrices
- time-dependent propagation
- coupling to nuclear dynamics

#### 3.7.2 `T` matrix

A matrix satisfying:

$$
\lvert \Psi \rangle
=
\sum_I C_I \lvert \text{CSF}_I \rangle
=
\sum_j
\left(
\sum_I T_{jI} C_I
\right)
\lvert D_j \rangle.
$$

This allows:

- switching between CSF and determinant representations
- enforcing spin purity during dynamics
- interfacing spin-adapted CI with determinant-based propagation


### 3.8. Example interpretation
[Back to TOC](#TOC)
<a name="3.8"></a>

For a singlet CAS with:

$$
S = 0, \quad M_s = 0,
$$

each CSF is a linear combination of multiple determinants with
paired $\alpha/\beta$ occupations.

The returned `mapped_basis` lists all such determinants, while `T`
encodes their **Clebsch–Gordan spin couplings**.


## 4. Worked Example: Determinants, CSFs, and the T Matrix (Simple Case)
[Back to TOC](#TOC)
<a name="4"></a>

Now, let's discuss what `configs_and_T_matrix` is doing, including **actual determinants, CSFs, and the
transformation matrix**.

We choose the *smallest nontrivial case* where spin adaptation matters.


### 4.1. System definition (toy model)
[Back to TOC](#TOC)
<a name="4.1"></a>

#### Active space
Two spatial orbitals:
$$
\mathcal{A} = \{1, 2\}
$$

Each generates two spin-orbitals:
$$
\{ +1, -1, +2, -2 \}
$$

#### Electrons
$$
N = 2
$$

#### Spin
Singlet state:
$$
S = 0, \quad M_s = 0
$$

This is the simplest case where **one CSF is a linear combination of multiple determinants**.

### 4.2. Determinant basis (fixed $M_s = 0$)
[Back to TOC](#TOC)
<a name="4.2"></a>

For $M_s = 0$, we require:
$$
N_\alpha = N_\beta = 1
$$

The allowed determinants are:

$$
\begin{aligned}
|D_1\rangle &= | +1,\,-1 \rangle \quad &(\text{double occupancy of orbital 1}) \\
|D_2\rangle &= | +2,\,-2 \rangle \quad &(\text{double occupancy of orbital 2}) \\
|D_3\rangle &= | +1,\,-2 \rangle \\
|D_4\rangle &= | -1,\,+2 \rangle
\end{aligned}
$$

These are already canonically ordered.


### 4.3. Spin-adapted CSFs
[Back to TOC](#TOC)
<a name="4.3"></a>

#### 4.3.1 Closed-shell singlets

Closed-shell determinants are already spin eigenfunctions:

$$
\begin{aligned}
|\text{CSF}_1\rangle &= | +1,\,-1 \rangle \\
|\text{CSF}_2\rangle &= | +2,\,-2 \rangle
\end{aligned}
$$


#### 4.3.2 Open-shell singlet CSF

The open-shell singlet is a **linear combination**:

$$
|\text{CSF}_3\rangle
=
\frac{1}{\sqrt{2}}
\Big(
| +1,\,-2 \rangle
-
| -1,\,+2 \rangle
\Big)
$$

This combination ensures:
$$
\hat{S}^2 |\text{CSF}_3\rangle = 0
$$


### 4.4. Summary of bases
[Back to TOC](#TOC)
<a name="4.4"></a>

#### 4.4.1. Determinant basis
$$
\{ |D_1\rangle, |D_2\rangle, |D_3\rangle, |D_4\rangle \}
$$

#### 4.4.2. CSF basis
$$
\{ |\text{CSF}_1\rangle, |\text{CSF}_2\rangle, |\text{CSF}_3\rangle \}
$$

Note:
- 4 determinants
- 3 singlet CSFs
- Determinants $\rightarrow$ CSFs is **many-to-one**


### 4.5. Configuration-to-CSF transformation matrix $T$
[Back to TOC](#TOC)
<a name="4.5"></a>

By definition:
$$
|\text{CSF}_I\rangle = \sum_j T_{jI} |D_j\rangle
$$

The matrix is:

$$
T =
\begin{pmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & \tfrac{1}{\sqrt{2}} \\
0 & 0 & -\tfrac{1}{\sqrt{2}}
\end{pmatrix}
$$

#### Interpretation
- Rows → determinants
- Columns → CSFs
- Each column is normalized
- Signs enforce fermionic antisymmetry and spin purity


### 4.6. Connection to `configs_and_T_matrix`
[Back to TOC](#TOC)
<a name="4.6"></a>

What the function does internally:

1. Generates all $M_s$-allowed determinants
2. Identifies linear combinations yielding total spin $S$
3. Constructs $T$ from Clebsch–Gordan coefficients
4. Maps orbital indices to `orbital_space`
5. Returns:
   - `mapped_basis` → determinants
   - `T` → spin-adaptation matrix

In this example:
- `mapped_basis` would be the four determinants
- `T` would be exactly the matrix shown above

<a name="configs_and_T_matrix-1"></a>

In [17]:
help(interfaces.configs_and_T_matrix)

Help on function configs_and_T_matrix in module libra_py.citools.interfaces:

configs_and_T_matrix(configs0_raw: List[Tuple[int, ...]], active_space: List[int], orbital_space: List[int], nelec: int, S: int, Ms: int) -> Tuple[List[Tuple[int, ...]], ForwardRef('CMATRIX')]
    Generate the minimal active-space configurations mapped to a given orbital space
    and the configuration-to-CSF transformation matrix for a CAS with given spin.
    
    Parameters
    ----------
    configs0_raw : list[tuple[int]]
        List of raw configurations from Libra/MOPAC (signed orbital indices).
    active_space : list[int]
        Orbitals defining the active space used to generate the minimal determinant basis.
    orbital_space : list[int]
        Orbital indices used for mapping configurations (output will be relative to this space).
    nelec : int
        Number of active electrons.
    S : int
        Total spin quantum number.
    Ms : int
        Spin projection quantum number.
    
    Retur

In [18]:
configs0_raw = [(6, -6, 7, -7, 9, -8), (6, -6, 7, -7, 10, -8)]
active_space = [6, 7, 8, 9, 10, 11]
orbital_space = [1,2,3,4,5,6,7,8,9,10,11]
nelec = 6
S = 0
Ms = 0
max_unpaired = 0 # singlets only

mapped_basis, T = interfaces.configs_and_T_matrix(
    configs0_raw,  active_space,  orbital_space,
    nelec, S, Ms)
print(T.num_of_rows, T.num_of_cols)
print(mapped_basis)

4 2
[(6, -6, 7, -7, 8, -9), (6, -6, 7, -7, -8, 9), (6, -6, 7, -7, 8, -10), (6, -6, 7, -7, -8, 10)]


In [19]:
T.show_matrix("T_orig.txt")

Analogously to above, we can generate these things much faster (especially for larger active spaces and many electrons) using the function below. This, however, works only for singlet excitations:

<a name="configs_and_T_matrix_singlet-1"></a>

In [20]:
help(interfaces.configs_and_T_matrix_singlet)

Help on function configs_and_T_matrix_singlet in module libra_py.citools.interfaces:

configs_and_T_matrix_singlet(configs0_raw: List[Tuple[int, ...]], active_space: List[int], orbital_space: List[int], nelec: int, S: int, Ms: int) -> Tuple[List[Tuple[int, ...]], ForwardRef('CMATRIX')]
    Generate the minimal active-space configurations mapped to a given orbital space
    and the configuration-to-CSF transformation matrix for a CAS with given spin.
    
    Parameters
    ----------
    configs0_raw : list[tuple[int]]
        List of raw configurations from Libra/MOPAC (signed orbital indices).
    active_space : list[int]
        Orbitals defining the active space used to generate the minimal determinant basis.
    orbital_space : list[int]
        Orbital indices used for mapping configurations (output will be relative to this space).
    nelec : int
        Number of active electrons.
    S : int
        Total spin quantum number.
    Ms : int
        Spin projection quantum number

In [21]:
# In this example, we have all MO time-overlaps (starting from 1 and beyond the orbitals of active space)
# so the mapped basis looks trivial (using the same indices as of the raw configurations)

configs0_raw = [(6, -6, 7, -7, 9, -8), (6, -6, 7, -7, 10, -8)]
active_space = [6, 7, 8, 9, 10, 11]         # list the indices of the spatial orbital (1-based)
                                            # across which the excitations are allowed
orbital_space = [1,2,3,4,5,6,7,8,9,10,11]   # list all the spatial orbitals (1-based) for which the
                                            # MO time-overlaps are available
nelec = 6                                   # how many electrons live in active space
S = 0
Ms = 0
max_unpaired = 0 # singlets only

mapped_basis, T = interfaces.configs_and_T_matrix_singlet(
    configs0_raw,  active_space,  orbital_space,
    nelec, S, Ms)
print(T.num_of_rows, T.num_of_cols)
print(mapped_basis)

4 2
[(6, -6, 7, -7, -8, 9), (6, -6, 7, -7, 8, -9), (6, -6, 7, -7, -8, 10), (6, -6, 7, -7, 8, -10)]


In [22]:
T.show_matrix("T.txt")

In [23]:
# In this example, we dropped MO time-overlaps below the active space - that is only active-space orbital
# overlaps are included - in this case, the mapped basis contains down-shifted indices

configs0_raw = [(6, -6, 7, -7, 9, -8), (6, -6, 7, -7, 10, -8)]

active_space = [6, 7, 8, 9, 10, 11]   # list the indices of the spatial orbital (1-based)
                                      # across which the excitations are allowed
orbital_space = [6, 7, 8, 9, 10,11]   # list all the spatial orbitals (1-based) for which the
                                      # MO time-overlaps are available
nelec = 6                             # how many electrons live in active space
S = 0
Ms = 0
max_unpaired = 0 # singlets only

mapped_basis, T = interfaces.configs_and_T_matrix_singlet(
    configs0_raw,  active_space,  orbital_space,
    nelec, S, Ms)
print(T.num_of_rows, T.num_of_cols)
print(mapped_basis)

4 2
[(1, -1, 2, -2, -3, 4), (1, -1, 2, -2, 3, -4), (1, -1, 2, -2, -3, 5), (1, -1, 2, -2, 3, -5)]
